## Shadow Fading — Stochastic Path Loss Variation

This demo introduces log-normal shadow fading as a stochastic component of the link budget.

In real environments, obstacles such as buildings, terrain and vegetation cause random
signal variations around the median path loss. This variability is modelled as a
zero-mean Gaussian random variable in dB, with standard deviation σ dependent on
the deployment scenario:

| Scenario | σ (dB) |
|----------|--------|
| Urban | 8 |
| Suburban | 6 |
| Rural | 4 |

A single shadowing realisation is compared against the deterministic baseline.
This demo motivates the need for Monte Carlo simulation — coming in v0.7 —
to statistically characterise coverage under shadowing conditions.

> Note: Shadow fading is applied over FSPL, consistent with the v0.3 implementation.
> In practice, shadowing is typically combined with empirical models such as
> Okumura-Hata or COST231 — this combination will be used in the Monte Carlo simulation of v0.7.


In [1]:
import sys
import os
from src.run_terrestrial import run_terrestrial
sys.path.insert(0, os.path.abspath('..'))
from params.tetra_config import tetra
import numpy as np
import  plotly.graph_objects as go

In [2]:
distance = np.linspace(0.1,30,300)
[margin_tetra, value0_tetra,rx_tetra] = run_terrestrial(tetra,distance,"fspl",False)
[margin_tetra_shadow, value0_tetra_shadow,rx_tetra_shadow] = run_terrestrial(tetra,distance,"fspl",True)

### Received Power vs Distance

The shadowed curve fluctuates around the deterministic baseline (thick red line).

Local variations can both improve and degrade received power at any given distance,

illustrating why a single deterministic link budget is insufficient to characterise

real-world coverage — especially near the coverage boundary.

In [3]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=distance, y=rx_tetra, name="TETRA 400MHz", line=dict(color="red", width=4)))

fig.add_trace(go.Scatter(x=distance, y=rx_tetra_shadow, name="TETRA 400MHz Shadowed", line=dict(color="blue",width=1)))

fig.update_layout(
    template='plotly_dark',
    title=dict(text ='Shadow effects', x=0.5),
    xaxis=dict(range=[0, 30]), xaxis_title='Distance (km)',
    yaxis=dict(range=[-90, -20]), yaxis_title='Rx power (dBm)',

)



### Link Margin vs Distance

Shadow fading introduces random fluctuations around the deterministic baseline.
With FSPL the link margin remains well above 0 dB across the full range —
the coverage boundary is never reached even with negative shadowing realisations.

This highlights a limitation of FSPL for shadowing analysis: the optimistic
path loss leaves insufficient margin variation to observe coverage degradation.
When applied over empirical models like Okumura-Hata or COST231 — as in v0.7 —
shadowing produces meaningful coverage boundary uncertainty and motivates
Monte Carlo simulation.

In [4]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=distance, y=margin_tetra, name="TETRA 400MHz", line=dict(color="red", width=4)))

fig.add_trace(go.Scatter(x=distance, y=margin_tetra_shadow, name="TETRA 400MHz Shadowed", line=dict(color="blue",width=1)))

fig.add_hline(
    y=0,
    line=dict(color='white', width=1.5, dash='dash'),
    annotation_text='Coverage threshold',
    annotation_position='bottom right'
)


fig.update_layout(
    template='plotly_dark',
    title=dict(text ='Shadow effects on link margin', x=0.5),
    xaxis=dict(range=[0, 30]), xaxis_title='Distance (km)',
    yaxis=dict(range=[-10, 80]), yaxis_title='Link Margin (dB)',

)
